In [5]:
import json
import pandas as pd

In [6]:
with open(
    "../resources/analyzer-results/greedy-phragmen-single-auto/metrics-greedy-phragmen-single-auto.json"
) as f:
    data = json.load(f)

metric_display_map = {
    "EXCLUSION_RATION": ("exclusion_ratio", "Exclusion Ratio"),
    "SUM_OBJECTIVES": ("sum", "Sum Objectives"),
    "TOTAL_COST": ("total_cost", "Total Cost"),
    "EJR_PLUS": ("ejr_plus", "EJR Plus"),
    "INSTANCE_SIZE": ("size", "Instance Size"),
}

processed = []
for entry in data:
    solver = entry.get("solver", "Unknown")
    options = entry.get("solver_options", {})
    solver_label = f"{solver}_{options}" if options else solver
    city = entry.get("city")

    for metric_key in entry.get("metrics", []):
        if metric_key in metric_display_map:
            val_key, display_name = metric_display_map[metric_key]
            val = entry.get(metric_key, {}).get(val_key)
            if val is not None:
                processed.append(
                    {
                        "Solver": solver_label,
                        "Metric": display_name,
                        "Value": val,
                        "City": city,
                    }
                )

    # instance size as metric row
    instance_size = entry.get("INSTANCE_SIZE", {}).get("size")
    if instance_size is not None:
        processed.append(
            {
                "Solver": solver_label,
                "Metric": "Instance Size",
                "Value": instance_size,
                "City": city,
            }
        )

    if "time" in entry:
        processed.append(
            {
                "Solver": solver_label,
                "Metric": "running time (s.)",
                "Value": entry["time"],
                "City": city,
            }
        )

df = pd.DataFrame(processed)
print(f"Loaded {len(data)} entries → {len(df)} rows")
print(f"Solvers: {df['Solver'].unique()}")
print(f"Metrics: {df['Metric'].unique()}")
print(f"Cities: {df['City'].nunique()}")
df.head(10)

Loaded 7139 entries → 49973 rows
Solvers: ['GREEDY' "PHRAGMEN_{'kappa': 1.0, 'increasing_scalings': True}"
 "PHRAGMEN_{'kappa': 0.0, 'increasing_scalings': False}"
 "PHRAGMEN_{'kappa': 0.0, 'increasing_scalings': True}"
 "PHRAGMEN_{'kappa': 1.0, 'increasing_scalings': False}"]
Metrics: ['Instance Size' 'Exclusion Ratio' 'Sum Objectives' 'Total Cost'
 'EJR Plus' 'running time (s.)']
Cities: 1428


,Solver,Metric,Value,City
0,GREEDY,Instance Size,9.000000e+00,poland_warszawa_2019_piaski
1,GREEDY,Exclusion Ratio,2.867384e-02,poland_warszawa_2019_piaski
2,GREEDY,Sum Objectives,1.955205e+08,poland_warszawa_2019_piaski
3,GREEDY,Total Cost,8.402480e+05,poland_warszawa_2019_piaski
4,GREEDY,EJR Plus,0.000000e+00,poland_warszawa_2019_piaski
5,GREEDY,Instance Size,9.000000e+00,poland_warszawa_2019_piaski
6,GREEDY,running time (s.),1.972079e-02,poland_warszawa_2019_piaski
7,GREEDY,Instance Size,8.000000e+00,poland_wroclaw_2016_rejon-nr-13-750
8,GREEDY,Exclusion Ratio,3.896848e-01,poland_wroclaw_2016_rejon-nr-13-750
9,GREEDY,Sum Objectives,9.585000e+08,poland_wroclaw_2016_rejon-nr-13-750


In [7]:
# Normalize Total Cost & Sum Objectives relative to GREEDY baseline per city
df_norm = df.copy()

for metric_label, norm_label in [
    ("Total Cost", "Total Cost (rel. to Greedy)"),
    ("Sum Objectives", "Sum Objectives (rel. to Greedy)"),
]:
    mask = df_norm["Metric"] == metric_label
    metric_df = df_norm[mask]
    greedy_baseline = metric_df[
        metric_df["Solver"].str.startswith("GREEDY")
    ].set_index("City")["Value"]
    df_norm.loc[mask, "Value"] = df_norm.loc[mask].apply(
        lambda row, bl=greedy_baseline: row["Value"] / bl[row["City"]]
        if row["City"] in bl.index and bl[row["City"]] != 0
        else row["Value"],
        axis=1,
    )
    df_norm.loc[mask, "Value"] = df_norm.loc[mask, "Value"].clip(upper=5.0)
    df_norm.loc[mask, "Metric"] = norm_label

print("Normalized metrics:")
print(df_norm["Metric"].value_counts())
df_norm.head(10)

Normalized metrics:
Metric
Instance Size                      14278
Exclusion Ratio                     7139
Sum Objectives (rel. to Greedy)     7139
Total Cost (rel. to Greedy)         7139
EJR Plus                            7139
running time (s.)                   7139
Name: count, dtype: int64


,Solver,Metric,Value,City
0,GREEDY,Instance Size,9.000000,poland_warszawa_2019_piaski
1,GREEDY,Exclusion Ratio,0.028674,poland_warszawa_2019_piaski
2,GREEDY,Sum Objectives (rel. to Greedy),1.000000,poland_warszawa_2019_piaski
3,GREEDY,Total Cost (rel. to Greedy),1.000000,poland_warszawa_2019_piaski
4,GREEDY,EJR Plus,0.000000,poland_warszawa_2019_piaski
5,GREEDY,Instance Size,9.000000,poland_warszawa_2019_piaski
6,GREEDY,running time (s.),0.019721,poland_warszawa_2019_piaski
7,GREEDY,Instance Size,8.000000,poland_wroclaw_2016_rejon-nr-13-750
8,GREEDY,Exclusion Ratio,0.389685,poland_wroclaw_2016_rejon-nr-13-750
9,GREEDY,Sum Objectives (rel. to Greedy),1.000000,poland_wroclaw_2016_rejon-nr-13-750


In [8]:
# Summary stats per metric
print("=== Raw data stats ===")
for metric in df["Metric"].unique():
    subset = df[df["Metric"] == metric]["Value"]
    print(f"\n--- {metric} ---")
    print(subset.describe())

print("\n=== Normalized data stats ===")
for metric in df_norm["Metric"].unique():
    subset = df_norm[df_norm["Metric"] == metric]["Value"]
    print(f"\n--- {metric} ---")
    print(subset.describe())

=== Raw data stats ===

--- Instance Size ---
count    14278.000000
mean        22.795349
std         25.896627
min          1.000000
25%          8.000000
50%         14.000000
75%         25.000000
max        220.000000
Name: Value, dtype: float64

--- Exclusion Ratio ---
count    7139.000000
mean        0.186259
std         0.188755
min         0.000000
25%         0.039446
50%         0.125893
75%         0.274347
max         0.973430
Name: Value, dtype: float64

--- Sum Objectives ---
count    7.139000e+03
mean     9.926259e+09
std      1.830757e+11
min      1.586000e+03
25%      1.242000e+08
50%      3.991875e+08
75%      1.170250e+09
max      6.904249e+12
Name: Value, dtype: float64

--- Total Cost ---
count    7.139000e+03
mean     2.577660e+06
std      2.837133e+07
min      8.000000e+00
25%      3.139655e+05
50%      5.890630e+05
75%      1.080000e+06
max      1.000000e+09
Name: Value, dtype: float64

--- EJR Plus ---
count    7139.000000
mean        0.489424
std         1.948

In [20]:
# Anomaly detection: normalized total cost > 1.0 means solver exceeded greedy baseline
anomalies = df_norm[
    (df_norm["Metric"] == "Total Cost (rel. to Greedy)")
    & (df_norm["Value"] > 1.10)
    & (~df_norm["Solver"].str.startswith("GREEDY"))
]
print(
    f"Found {len(anomalies)} rows where solver total cost exceeds greedy baseline"
)
anomalies.sort_values("Value", ascending=False)

Found 12 rows where solver total cost exceeds greedy baseline


,Solver,Metric,Value,City
11826,"PHRAGMEN_{'kappa': 0.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),5.000000,poland_krakow_2020_wzgorza-krzeslawickie
32049,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),5.000000,poland_krakow_2020_wzgorza-krzeslawickie
37782,"PHRAGMEN_{'kappa': 0.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),5.000000,poland_krakow_2020_wzgorza-krzeslawickie
43508,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),5.000000,poland_krakow_2020_wzgorza-krzeslawickie
1333,"PHRAGMEN_{'kappa': 0.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.179671,poland_katowice_2023_zaleska-halda-brynow-czes...
10888,"PHRAGMEN_{'kappa': 0.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.179671,poland_katowice_2023_zaleska-halda-brynow-czes...
22445,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.179671,poland_katowice_2023_zaleska-halda-brynow-czes...
44712,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.179671,poland_katowice_2023_zaleska-halda-brynow-czes...
4035,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.121875,poland_poznan_2025_6-ogrody-solacz-winiary-wol...
5547,"PHRAGMEN_{'kappa': 1.0, 'increasing_scalings':...",Total Cost (rel. to Greedy),1.121875,poland_poznan_2025_6-ogrody-solacz-winiary-wol...


In [21]:
anomalies["City"].unique()

array(['poland_katowice_2023_zaleska-halda-brynow-czesc-zachodnia',
       'poland_poznan_2025_6-ogrody-solacz-winiary-wola-large',
       'poland_krakow_2020_wzgorza-krzeslawickie'], dtype=object)

In [10]:
# Filter helpers — adjust these to slice data interactively
def filter_df(source_df, cities=None, solvers=None, metrics=None):
    """Filter dataframe by city, solver, metric lists."""
    result = source_df.copy()
    if cities:
        result = result[result["City"].isin(cities)]
    if solvers:
        result = result[result["Solver"].isin(solvers)]
    if metrics:
        result = result[result["Metric"].isin(metrics)]
    return result


# Example usage:
# filtered = filter_df(df_norm, cities=["poland_warszawa_2019_piaski"], solvers=["GREEDY"])
# filtered = filter_df(df, metrics=["Total Cost", "Instance Size"])
print("Available solvers:", df["Solver"].unique().tolist())
print("Available metrics:", df["Metric"].unique().tolist())
print(f"Available cities: {df['City'].nunique()} total")

Available solvers: ['GREEDY', "PHRAGMEN_{'kappa': 1.0, 'increasing_scalings': True}", "PHRAGMEN_{'kappa': 0.0, 'increasing_scalings': False}", "PHRAGMEN_{'kappa': 0.0, 'increasing_scalings': True}", "PHRAGMEN_{'kappa': 1.0, 'increasing_scalings': False}"]
Available metrics: ['Instance Size', 'Exclusion Ratio', 'Sum Objectives', 'Total Cost', 'EJR Plus', 'running time (s.)']
Available cities: 1428 total
